## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

> ### How to run this notebook (read me first)
>
> This project loads a **7-billion-parameter LLM** and embeds a **4,000+ page PDF**, so it needs a machine with a **GPU** and a few GB of free disk space.
>
> **Recommended:** Google Colab with a GPU runtime (`Runtime -> Change runtime type -> T4 GPU`).
>
> **Run order:**
> 1. Run the GPU install cell, then **restart the runtime** when asked.
> 2. Run the library install cell, then **restart the runtime** again.
> 3. From the imports cell onward, run every cell **top to bottom**.
> 4. Upload the PDF and update the `pdf_path` variable in the *Loading the Data* section.
>
> Building the vector database over the full manual takes a few minutes the first time. After running everything, export to HTML for submission (`File -> Download -> .html`, or `jupyter nbconvert --to html`).

## Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 56.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 290.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 272.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 307.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.0 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 fastembed -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.6/486.6 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [4]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [5]:
# =====================================================================
# STEP 1 - Download the Large Language Model (LLM) from Hugging Face
# =====================================================================
# We use Mistral-7B-Instruct in the "GGUF" format. GGUF is a compressed
# file format that lets a 7-billion-parameter model run on a single GPU
# (or even a CPU) using the llama-cpp library.
# "Q4_K_M" means the weights are quantized (compressed) to 4 bits, which
# keeps the download small (~4 GB) while keeping the answers good.

model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"   # the repo on Hugging Face
model_file = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"      # the exact file we want

# hf_hub_download() downloads the file once and caches it locally.
# It returns the full path to the file on disk.
model_path = hf_hub_download(repo_id=model_name, filename=model_file)
print("Model downloaded to:\n", model_path)


# =====================================================================
# STEP 2 - Load the model into memory with llama-cpp
# =====================================================================
llm = Llama(
    model_path=model_path,   # location of the .gguf file downloaded above
    n_ctx=2048,              # context window = max tokens (prompt + answer)
    n_gpu_layers=-1,         # -1 puts ALL layers on the GPU; use 0 for CPU-only
    n_batch=512,             # number of tokens processed in parallel
    verbose=True            # keep the output clean (hide internal logs)
)
print("\nModel loaded and ready to answer questions!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

Model downloaded to:
 /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q4_K_M.gguf

Model loaded and ready to answer questions!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


#### Response

In [6]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [7]:
# We define the five business questions ONCE here and reuse them everywhere.
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

# Ask the base model (no extra context, no special prompt).
# max_tokens=512 lets the model write a reasonably complete answer.
print(response(query_1, max_tokens=512))



Sepsis is a life-threatening condition that can arise from an infection, and prompt recognition and appropriate management are crucial for improving outcomes. In a critical care unit, the following steps should be taken for managing sepsis:

1. Early recognition: Identify patients at risk of developing sepsis based on clinical signs, laboratory values, and medical history. Use validated scoring systems such as Sequential Organ Failure Assessment (SOFA) or Quick Sequential Organ Failure Assessment (qSOFA) to help identify suspected sepsis.
2. Immediate resuscitation: Begin fluid resuscitation with isotonic crystalloids to maintain adequate tissue perfusion and organ function. Monitor hemodynamic parameters such as mean arterial pressure, central venous pressure, and urine output. Consider the use of vasopressors or inotropes if necessary to maintain adequate blood pressure and organ perfusion.
3. Source control: Identify and address the source of infection promptly. This may involve s

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [8]:
print(response(query_2, max_tokens=512))

Llama.generate: prefix-match hit




Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain may start as a mild discomfort in the lower right side of the abdomen and gradually worsen over time. It is often described as a dull ache that becomes sharp and localized.
2. Loss of appetite: People with appendicitis may lose their appetite due to abdominal pain or nausea.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially if the inflammation is severe.
4. Fever: A fever above 101°F (38.3°C) may be present in some cases.
5. Constipation or diarrhea: Both constipation and diarrhea can occur with appendicitis, depending on the location of the inflammation and the pressure it puts on the intestines.
6. Abdominal swelling: The abdomen may become swollen and tender to the touch.
7. In

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [9]:
print(response(query_3, max_tokens=512))

Llama.generate: prefix-match hit




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can cause round or oval bald patches on the scalp, beard, or other areas of the body where hair grows. The exact cause of alopecia areata is not known, but it's believed to be related to an abnormal immune response that attacks the hair follicles.

There are several treatments and solutions for addressing sudden patchy hair loss:

1. Corticosteroids: These medications can help reduce inflammation and suppress the immune system, allowing the hair to grow back. They can be applied topically or taken orally.
2. Immunotherapy: This involves using drugs that alter the immune response to promote hair growth. One such drug is minoxidil, which has been shown to be effective in some cases of alopecia areata.
3. Hair transplantation: In severe cases where there is extensive hair loss, hair transplantation may be an option. This involves taking healthy hair follicles from 

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [10]:
print(response(query_4, max_tokens=512))

Llama.generate: prefix-match hit




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek immediate medical attention. The primary goal of emergency care is to prevent further damage, stabilize the patient's condition, and ensure adequate oxygen supply to the brain.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage conditions such as seizures, pain, swelling, or infections. For instance, anti-inflammatory drugs, anticonvulsants, or steroids might be used.
3. Rehabilitation therapy: Rehabilitation therapies are essential for individuals with brain injuries to help them regain lost skills and improve their overall functioning. Some common rehabilitation therapies include physical therapy, occupational therapy, speech-language therap

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [11]:
print(response(query_5, max_tokens=512))

Llama.generate: prefix-match hit




First and foremost, it is essential to ensure the safety of the injured person. If possible, try to keep them calm and still to prevent further injury or discomfort. If the fracture is open (compound), take extra precautions to prevent infection by covering the wound with a clean dressing.

Next, assess the severity of the fracture and determine if medical attention is required. If the person can walk with assistance or if the fracture is not displaced (meaning the bone ends are still aligned), they may be able to hike out with help. However, if the fracture is severe, displaced, or open, it is crucial to seek medical attention as soon as possible.

If medical attention is not immediately available, follow these steps for temporary care and stabilization:

1. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg above and below the fracture site. This will help prevent further injury and discomfort.
2. Elevate the leg: Try to keep the injured leg

### Observations - Question Answering using the base LLM
- The model answers **from memory only** - it never saw the Merck Manual, so it relies on whatever medical knowledge was in its training data.
- Answers are usually fluent and broadly sensible, but they carry two real risks for healthcare: the model can **hallucinate** (state confident but incorrect facts) and it **cannot cite a trusted source**.
- There is no way to verify an answer against an authoritative manual, which is unacceptable for clinical decisions.
- **Takeaway:** a raw LLM is a helpful starting point but not safe to deploy as-is. The next steps (prompt engineering, then RAG) make the answers more controlled and more trustworthy.

## Question Answering using LLM with Prompt Engineering

Below we define a small helper that wraps each question in a **system message** and Mistral's required `[INST] ... [/INST]` prompt format, then we try **5 different parameter combinations**.

In [12]:
# ---------------------------------------------------------------------
# Helper that adds (1) a clear ROLE / SYSTEM instruction and
# (2) the special prompt format that Mistral-Instruct expects.
# Mistral wraps instructions inside  <s>[INST] ... [/INST]  tags.
# ---------------------------------------------------------------------
def response_with_prompt(query, system_message,
                         max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    prompt = f"<s>[INST] {system_message}\n\nQuestion: {query} [/INST]"
    output = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )
    return output["choices"][0]["text"].strip()


# A reusable system message that tells the model exactly how to behave.
system_message = (
    "You are a careful, knowledgeable medical assistant. "
    "Answer the question clearly and concisely using correct medical terminology. "
    "Organise the answer into short, easy-to-read points. "
    "If you are not certain about something, say so instead of guessing."
)

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
# Combination 1 -> fully deterministic (temperature=0): the most factual setting.
print(response_with_prompt(query_1, system_message,
                           temperature=0.0, top_p=0.90, top_k=50, max_tokens=512))

Llama.generate: prefix-match hit


Sepsis is a life-threatening condition caused by the body's response to an infection. In a critical care unit, managing sepsis involves the following steps:

1. Early recognition and diagnosis: Recognize sepsis early based on clinical suspicion and laboratory findings such as elevated white blood cell count, lactate level, and procalcitonin. Use the Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score to assess organ dysfunction.
2. Fluid resuscitation: Administer intravenous fluids to maintain adequate tissue perfusion and restore intravascular volume. Aim for a mean arterial pressure (MAP) of 65 mmHg or higher and a central venous oxygen saturation (ScvO2) of >70%.
3. Vasopressor support: Use vasopressors to maintain MAP if fluid resuscitation is insufficient. Norepinephrine is the first-line vasopressor in sepsis.
4. Antibiotics: Administer broad-spectrum antibiotics as soon as possible based on culture and sensitivity results or clinical suspicion.
5. Corticosteroids: 

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:
# Combination 2 -> a little randomness (temperature=0.3) for more natural wording.
print(response_with_prompt(query_2, system_message,
                           temperature=0.3, top_p=0.95, top_k=40, max_tokens=512))

Llama.generate: prefix-match hit


Appendicitis is an inflammatory condition of the vermiform appendix, a small worm-shaped pouch that extends from the caecum in the large intestine. Here are some common symptoms of appendicitis:

1. Abdominal pain: Appendingitis typically starts as a pain around the navel or lower right abdomen, which may be mild at first but becomes sharp and localized in the right lower quadrant as the inflammation progresses.
2. Loss of appetite: A decrease or complete loss of appetite is common due to the discomfort and nausea associated with appendicitis.
3. Nausea and vomiting: Vomiting may occur, especially if there is an obstruction in the intestine.
4. Fever: A low-grade fever (100.4°F or 38°C) is common as the body responds to the infection.
5. Constipation or diarrhea: Depending on the stage of appendicitis, a person may experience constipation or diarrhea.
6. Abdominal swelling: The abdomen can become tender and swollen due to inflammation.
7. Inability to pass gas: Passing gas is often dif

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
# Combination 3 -> medium creativity (temperature=0.5), narrower top_k.
print(response_with_prompt(query_3, system_message,
                           temperature=0.5, top_p=0.90, top_k=30, max_tokens=512))

Llama.generate: prefix-match hit


1. Causes of Sudden Patchy Hair Loss:
   a. Alopecia Areata: An autoimmune disorder that results in sudden hair loss from specific areas of the scalp.
   b. Telogen Effluvium: A condition where a large number of hairs enter the resting phase, causing shedding.
   c. Trauma or Stress: Physical or emotional stress can cause sudden hair loss.
   d. Nutritional Deficiencies: Lack of iron, protein, or vitamins can lead to patchy hair loss.
   e. Hormonal Imbalance: Hormonal changes due to pregnancy, menopause, or thyroid issues can cause hair loss.

2. Effective Treatments for Sudden Patchy Hair Loss:
   a. Alopecia Areata: Corticosteroids (topical or injectable) are the most common treatment. Other options include immunotherapy, minoxidil, and wigs.
   b. Telogen Effluvium: Treating the underlying cause, such as nutritional deficiencies or hormonal imbalances, can help resolve the issue. Minoxidil may also be used to promote hair growth.
   c. Trauma or Stress: Addressing the underlying ca

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
# Combination 4 -> higher creativity (temperature=0.7); good to test stability.
print(response_with_prompt(query_4, system_message,
                           temperature=0.7, top_p=0.85, top_k=50, max_tokens=512))

Llama.generate: prefix-match hit


1. Traumatic Brain Injury (TBI) treatment depends on the severity and location of the injury.
2. Immediate intervention: Airway management, breathing support, and circulation stabilization are crucial.
3. Medications: Anti-inflammatory drugs, anticonvulsants, and diuretics may be prescribed to manage swelling, seizures, or increased intracranial pressure.
4. Surgical interventions: Craniotomy, decompressive craniectomy, or evacuation of hematomas might be necessary to remove pressure on the brain.
5. Rehabilitation: Physical therapy, occupational therapy, speech and language therapy, and cognitive rehabilitation can help improve function and independence.
6. Supportive care: Nutritional support, hydration, and psychological counseling are essential for overall well-being.
7. Assistive devices: Devices like wheelchairs, walkers, or prosthetics may be necessary to aid mobility and daily living activities.
8. Lifestyle modifications: Adjustments in diet, exercise, sleep patterns, and stre

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
# Combination 5 -> low temperature but a longer answer (max_tokens=600).
print(response_with_prompt(query_5, system_message,
                           temperature=0.2, top_p=0.95, top_k=20, max_tokens=600))

Llama.generate: prefix-match hit


1. Assess the Severity: Evaluate the extent of the leg fracture by determining the location, type (open or closed), and severity of the fracture. This information will guide the necessary treatment steps.
2. Immobilize the Fracture: Apply a splint, sling, or cast to immobilize the affected limb and prevent further damage. This will help reduce pain, swelling, and promote proper healing.
3. Pain Management: Administer appropriate pain relief medication as prescribed by a healthcare professional to manage discomfort and improve mobility during recovery.
4. Transportation: Arrange for transportation to a medical facility as soon as possible for further evaluation and treatment. If transporting the person yourself, ensure they are comfortable, maintain body temperature, and provide hydration.
5. Tetanus Shot: Ensure the individual has received a tetanus shot within the past ten years or receives one if the fracture is an open (compound) fracture, as this can help prevent tetanus infection.

### Observations - Prompt Engineering & parameter tuning
We tested **5 parameter combinations** together with a clear *system message* that gives the model a role and formatting rules.
- Adding a **system message** ("you are a careful medical assistant... say so if unsure") makes answers more structured and more cautious.
- **temperature = 0** gives the most factual, repeatable answers - best for medical content. As temperature rises (0.3 -> 0.7) wording becomes more varied but the risk of drifting or inventing detail grows.
- **top_p** and **top_k** narrow the words the model may choose from; lower values keep it focused.
- **Limitation:** even the best prompt cannot fix the core problem - the model still answers from memory and can still hallucinate. To truly ground answers in a trusted source we need **RAG**.

## Data Preparation for RAG

### Loading the Data

In [19]:
# ---------------------------------------------------------------------
# Load the Merck Manual PDF.
# PyMuPDFLoader returns a LIST of documents - one document per PDF page.
# ---------------------------------------------------------------------
pdf_path = "/content/drive/MyDrive/medical_diagnosis_manual_opt_opt.pdf"


loader = PyMuPDFLoader(pdf_path)
data = loader.load()

print("Number of pages loaded:", len(data))

Number of pages loaded: 4114


### Data Overview

#### Checking the first 5 pages

In [20]:
# Peek at the first 5 pages so we can see what the raw text looks like.
for page in data[:5]:
    print("----- Page", page.metadata.get("page"), "-----")
    print(page.page_content[:500])   # show only the first 500 characters
    print()

----- Page 0 -----
dave.idonije@gmail.com
K4E0DVCSQT
This file is meant for personal use by dave.idonije@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

----- Page 1 -----
dave.idonije@gmail.com
K4E0DVCSQT
This file is meant for personal use by dave.idonije@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.

----- Page 2 -----
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................

----- Page 3 -----
491
Chapter 44. Foot & Ankle Disorders    ................

#### Checking the number of pages

In [21]:
print("Total number of pages in the manual:", len(data))

Total number of pages in the manual: 4114


### Data Chunking

In [22]:
# ---------------------------------------------------------------------
# Chunking = cutting the long text into small, overlapping pieces.
# Why? A 4,000-page book is far too big to feed to the model at once.
# We retrieve only the few small chunks that match a question.
#   chunk_size    -> how many characters per chunk
#   chunk_overlap -> characters shared between neighbours (keeps context
#                    from being cut off mid-sentence)
# ---------------------------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]   # try to split on natural breaks first
)

chunks = text_splitter.split_documents(data)

print("Original pages :", len(data))
print("Total chunks   :", len(chunks))
print("\nExample chunk:\n", chunks[0].page_content[:300])

Original pages : 4114
Total chunks   : 16586

Example chunk:
 dave.idonije@gmail.com
K4E0DVCSQT
This file is meant for personal use by dave.idonije@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.


### Embedding

In [23]:
# ---------------------------------------------------------------------
# An embedding model turns text into a list of numbers (a "vector").
# Texts with similar meaning get similar vectors, which is how we later
# find the chunks that match a question.
#
# We use FastEmbed here. It runs the model with ONNX, so it does NOT need
# torch / sentence-transformers / scipy. That avoids the numpy/scipy
# version clash (the "_center from numpy._core.umath" error) and is faster.
# "BAAI/bge-small-en-v1.5" is a small, fast, high-quality English model.
#
# We try to embed on the GPU (CUDA) for speed, and automatically fall back
# to the CPU if no GPU is present or this FastEmbed version is older.
# ---------------------------------------------------------------------
try:
    embedding_model = FastEmbedEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        providers=["CUDAExecutionProvider", "CPUExecutionProvider"],  # GPU first
    )
    print("Embedding model loaded (will use GPU if available).")
except Exception as e:
    print("GPU providers unavailable, using CPU instead. Reason:", e)
    embedding_model = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
    print("Embedding model loaded (CPU).")

# Quick check that it works (prints the length of one embedding vector).
print("Vector length:", len(embedding_model.embed_query("test sentence")))

GPU providers unavailable, using CPU instead. Reason: No package metadata was found for fastembed-gpu


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

model_optimized.onnx:   0%|          | 0.00/66.5M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Embedding model loaded (CPU).
Vector length: 384


### Vector Database

In [24]:
# ---------------------------------------------------------------------
# Build the vector database in small BATCHES that save to disk as we go.
#
# Why batches? A 4,000-page manual produces tens of thousands of chunks,
# and embedding them all in one shot can run for many minutes. If Colab
# disconnects mid-way, a single-shot build loses EVERYTHING. With batches,
# progress is written to disk after each batch - so if you get
# disconnected you simply RE-RUN this cell and it CONTINUES from where it
# stopped instead of starting over.
# ---------------------------------------------------------------------
from tqdm import tqdm   # simple text progress bar (safe for HTML export)

PERSIST_DIR = "./merck_manual_db"   # folder on disk where the database lives
BATCH_SIZE  = 200                   # how many chunks to embed per batch

# Open the database on disk (creates it the first time, reopens it later).
vector_store = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding_model,
)

# How many chunks are ALREADY saved? This is what lets us resume.
already_saved = vector_store._collection.count()
print(f"Chunks already in the database: {already_saved} / {len(chunks)}")

# Add only the remaining chunks, one batch at a time.
for start in tqdm(range(already_saved, len(chunks), BATCH_SIZE)):
    batch = chunks[start:start + BATCH_SIZE]
    vector_store.add_documents(batch)          # embed + store this batch
    # Newer Chroma auto-saves; older versions need .persist(). The
    # try/except keeps the cell working on either version.
    try:
        vector_store.persist()
    except Exception:
        pass

print("Done! Vector database now holds",
      vector_store._collection.count(), "vectors.")

/tmp/ipykernel_2870/953534157.py:17: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


Chunks already in the database: 0 / 16586


  0%|          | 0/83 [00:00<?, ?it/s]/tmp/ipykernel_2870/953534157.py:33: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()
100%|██████████| 83/83 [1:06:14<00:00, 47.88s/it]

Done! Vector database now holds 16586 vectors.


> **Note on Colab disconnections:** the cell above builds the database in **batches and saves to disk after each one**. If Colab disconnects while it runs, just **re-run the same cell** - it checks how many chunks are already stored and continues from there instead of starting over. (Tip from the FAQ: if it's still slow, a larger `chunk_size` or smaller `chunk_overlap` makes fewer chunks to embed.)

### Retriever

In [25]:
# ---------------------------------------------------------------------
# The retriever is the search engine on top of the vector database.
#   search_type="similarity" -> return the closest chunks by meaning
#   k=3                       -> return the top 3 most relevant chunks
# ---------------------------------------------------------------------
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
print("Retriever ready (returns top 3 chunks per question).")

Retriever ready (returns top 3 chunks per question).


### System and User Prompt Template

In [26]:
# ---------------------------------------------------------------------
# Prompt templates used by the RAG response function.
#
# qna_system_message: the rules. The most important rule is that the
# model must answer ONLY from the retrieved context. This is what stops
# the model from "hallucinating" (making up) medical facts.
#
# qna_user_message_template: where the retrieved {context} and the
# user's {question} get inserted at run time.
# ---------------------------------------------------------------------
qna_system_message = """You are a helpful medical assistant for healthcare professionals.
Use ONLY the information in the provided context to answer the user's question.
Guidelines:
- Base every statement strictly on the context. Do not use outside knowledge.
- If the answer is not contained in the context, reply exactly:
  "I do not have enough information in the manual to answer this question."
- Be clear and concise, and use correct medical terminology.
"""

qna_user_message_template = """###Context
Here is the context you must use to answer the question:
{context}

###Question
{question}
"""

print("Prompt templates are set.")

Prompt templates are set.


### Response Function

In [27]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [28]:
print(generate_rag_response(query_1, k=3, max_tokens=512))

/tmp/ipykernel_2870/141734326.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
Llama.generate: prefix-match hit


###Answer
The protocol for managing sepsis in a critical care unit involves the following steps based on the context provided:
1. Suspect sepsis when a patient with a known infection develops systemic signs of inflammation or organ dysfunction, or when a patient presents with unexplained signs of systemic inflammation.
2. Evaluate for infection by obtaining a thorough history and physical examination, as well as performing tests such as urinalysis and urine culture (especially in patients with indwelling catheters), serial blood cultures, and cultures of other suspect body fluids.
3. Monitor and treat physiologic parameters, including adequate nutrition, prevention of infection, stress ulcers and gastritis, and pulmonary embolism.
4. Diagnose sepsis clinically based on symptoms and test results.
5. Rule out other causes of shock such as hypovolemia, myocardial infarction, or hypoperfusion.
6. Provide aggressive fluid resuscitation, administer antibiotics, perform surgical excision of i

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [29]:
print(generate_rag_response(query_2, k=3, max_tokens=512))

Llama.generate: prefix-match hit


###Answer
The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which later shifts to the right lower quadrant. The pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point. Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant. Appendicitis cannot be cured via medicine alone, as it requires surgical removal. The treatment for acute appendicitis is open or laparoscopic appendectomy. Antibiotics should be administered prior to surgery.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [30]:
print(generate_rag_response(query_3, k=3, max_tokens=512))

Llama.generate: prefix-match hit


Based on the context provided, the condition being described is Alopecia Areata. The effective treatments or solutions for addressing sudden patchy hair loss, which is a common symptom of Alopecia Areata, include:
1. Topical corticosteroids
2. Intralesional corticosteroids
3. Systemic corticosteroids (in severe cases)
4. Topical minoxidil
5. Topical anthralin
6. Topical immunotherapy (diphencyprone or squaric acid dibutylester)
7. Psoralen plus ultraviolet A (PUVA)
8. Elimination of physical traction or stress to the scalp (for traction alopecia)
9. Topical or oral antifungals (for tinea capitis)
10. Behavior modification, clomipramine, or an SSRI for Trichotillomania
11. Long-acting oral tetracycline in combination with a potent topical corticosteroid for scarring alopecia
12. Oral antimalarials, corticosteroids, retinoids, or immunosuppressants for Lichen planopilaris and chronic cutaneous lupus lesions
It is important to note that the causes of Alopecia Areata are not fully understo

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [31]:
print(generate_rag_response(query_4, k=3, max_tokens=512))

Llama.generate: prefix-match hit


###Answer
Based on the context provided, the recommended treatments for a person with a brain injury include: immobilization, maintenance of oxygenation and perfusion, supportive care, sometimes surgical stabilization, possibly methylprednisolone for blunt injuries, long-term symptomatic care, rehabilitation through a team approach, and planning for early rehabilitation services. The goal is to prevent secondary injury and provide necessary care and support for the patient's recovery. Rehabilitation is particularly important when neurologic deficits persist.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [32]:
print(generate_rag_response(query_5, k=3, max_tokens=512))

Llama.generate: prefix-match hit


Based on the context provided, here is the answer:

For a person who has fractured their leg during a hiking trip, the following precautions and treatment steps are necessary:

1. Assess the severity of the injury: Determine if it's an open or closed fracture, and evaluate for any signs of complications such as rapid blood loss, fat embolism, or ischemia (sudden, severe pain, palpable defect along the tendon, positive Thompson test).
2. Control bleeding: Apply direct pressure to the wound if it's open, and elevate the affected limb to minimize swelling and reduce bleeding.
3. Immobilize the fracture: Use a splint or cast to keep the bone in place and prevent further damage.
4. Pain management: Administer pain medication as needed to help manage discomfort.
5. Rest and avoid weight-bearing activities: Encourage the person to rest and avoid putting weight on the affected leg until the fracture has healed.
6. Monitor for complications: Keep a close eye on the person for any signs of infec

### Observations - Question Answering using RAG
- Each answer is now built **only from chunks retrieved from the Merck Manual**, so responses are grounded in a trusted, authoritative source.
- Hallucination drops sharply: the system message forces the model to answer from the supplied context and to say *"I do not have enough information..."* when the manual does not cover the question.
- Compared with the base LLM, RAG answers are more specific, more consistent, and **traceable** back to the source material - exactly what a clinical setting requires.
- Answer quality now depends on **retrieval quality**: if the right chunks are not retrieved, the answer suffers. That is why we tune the retriever next.

### Fine-tuning

In [33]:
# ---------------------------------------------------------------------
# Fine-tuning: vary CHUNKING + RETRIEVER + LLM across at least 5 combos.
# Changing chunk_size/overlap means re-splitting the text and building
# another vector store. To keep this feasible and disconnect-proof we:
#   (a) build each chunk setting only ONCE,
#   (b) build it in resumable batches (re-run safe if Colab disconnects),
#   (c) REUSE the store we already built above for the default 1000/100.
# (This is the slowest cell - please be patient.)
# ---------------------------------------------------------------------
from tqdm import tqdm

def build_or_load_store(chunk_size, chunk_overlap, batch_size=200):
    """Split with these settings and build/resume a Chroma store on disk."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    these_chunks = splitter.split_documents(data)
    store = Chroma(
        persist_directory=f"./merck_db_{chunk_size}_{chunk_overlap}",
        embedding_function=embedding_model,
    )
    saved = store._collection.count()        # resume support
    for start in tqdm(range(saved, len(these_chunks), batch_size),
                      desc=f"chunks {chunk_size}/{chunk_overlap}"):
        store.add_documents(these_chunks[start:start + batch_size])
        try:
            store.persist()
        except Exception:
            pass
    return store, len(these_chunks)


# The 5 combinations to try (chunking + retriever k + LLM temperature/length).
combinations = [
    {"chunk_size": 1000, "chunk_overlap": 100, "k": 3, "temperature": 0.0, "max_tokens": 512},
    {"chunk_size": 1000, "chunk_overlap": 100, "k": 5, "temperature": 0.0, "max_tokens": 512},
    {"chunk_size": 1000, "chunk_overlap": 100, "k": 3, "temperature": 0.5, "max_tokens": 512},
    {"chunk_size": 1500, "chunk_overlap": 150, "k": 3, "temperature": 0.0, "max_tokens": 512},
    {"chunk_size": 1500, "chunk_overlap": 150, "k": 4, "temperature": 0.3, "max_tokens": 600},
]

# Seed the cache with the store we ALREADY built (1000/100) so we don't redo it.
store_cache = {(1000, 100): (vector_store, len(chunks))}
test_query = query_1   # use one question so the combinations are comparable

for i, combo in enumerate(combinations, start=1):
    chunk_key = (combo["chunk_size"], combo["chunk_overlap"])
    if chunk_key not in store_cache:                 # build new chunk settings once
        store_cache[chunk_key] = build_or_load_store(*chunk_key)
    store, n_chunks = store_cache[chunk_key]

    # Point the (global) retriever at this store with this combo's k value.
    retriever = store.as_retriever(search_type="similarity",
                                   search_kwargs={"k": combo["k"]})

    print("=" * 72)
    print(f"Combination {i}: {combo}   (total chunks in store: {n_chunks})")
    print("=" * 72)
    print(generate_rag_response(test_query, k=combo["k"],
                                temperature=combo["temperature"],
                                max_tokens=combo["max_tokens"]), "\n")

# Reset the retriever to the default setting for the rest of the notebook.
retriever = vector_store.as_retriever(search_type="similarity",
                                      search_kwargs={"k": 3})

Combination 1: {'chunk_size': 1000, 'chunk_overlap': 100, 'k': 3, 'temperature': 0.0, 'max_tokens': 512}   (total chunks in store: 16586)


Llama.generate: prefix-match hit


###Answer
The protocol for managing sepsis in a critical care unit involves the following steps based on the context provided:
1. Suspect sepsis when a patient with a known infection develops systemic signs of inflammation or organ dysfunction, or when a patient presents with unexplained signs of systemic inflammation.
2. Evaluate for infection by obtaining a thorough history and physical examination, as well as performing tests such as urinalysis and urine culture (especially in patients with indwelling catheters), serial blood cultures, and cultures of other suspect body fluids.
3. Monitor and treat physiologic parameters, including adequate nutrition, prevention of infection, stress ulcers and gastritis, and pulmonary embolism.
4. Diagnose sepsis clinically based on symptoms and test results.
5. Rule out other causes of shock such as hypovolemia, myocardial infarction, or hypoperfusion.
6. Provide aggressive fluid resuscitation, administer antibiotics, perform surgical excision of i

Llama.generate: prefix-match hit


Answer:
The protocol for managing sepsis in a critical care unit involves aggressive supportive care, including adequate nutrition, prevention of infection, stress ulcers and gastritis, and pulmonary embolism. Diagnosis of sepsis is clinical, based on systemic signs of inflammation or organ dysfunction. Treatment includes fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues, drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Patients with septic shock should be treated in an ICU and monitored frequently for systemic pressure, central venous pressure (CVP), pulmonary artery occlusive pressure (PAOP), pulse oximetry, arterial blood gases (ABGs), blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2. Fluid resuscitation with 0.9% saline should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. Oliguria 

...

Based on the context, the protocol for managing sepsis in a critical care unit includes:
1. Suspected sepsis should be diagnosed by evaluating patients with known infections who develop systemic signs of inflammation or organ dysfunction, or patients with unexplained signs of systemic inflammation who require evaluation for infection through history, physical examination, and tests such as urinalysis, urine culture, serial blood cultures, and cultures of other suspect body fluids.
2. Diagnosis is clinical, but elevated levels of procalcitonin and C-reactive protein may facilitate diagnosis. Other causes of shock should also be sought via history, physical examination, ECG, and serum cardiac markers.
3. Treatment includes aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues and drainage of pus, supportive care, intensive control of blood glucose, administration of corticosteroids, and activated protein C if necessary.
4. Prevention of infection,

chunks 1500/150: 100%|██████████| 60/60 [1:15:31<00:00, 75.53s/it]
Llama.generate: prefix-match hit


Combination 4: {'chunk_size': 1500, 'chunk_overlap': 150, 'k': 3, 'temperature': 0.0, 'max_tokens': 512}   (total chunks in store: 11842)
###Answer
The management of sepsis in a critical care unit involves aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. The patient's vital signs, including temperature, blood pressure, pulse, and respiration rate, should be monitored closely, along with quantification of all fluid intake and output and daily weight. Symptoms such as shaking chills, fever, hypotension, oliguria, and confusion indicate the presence of sepsis, severe sepsis, or septic shock. Treatment is initiated based on these symptoms and laboratory findings, which may include an elevated heart rate, respiratory rate, or white blood cell count. Risk-benefit assessment is required for patients 

Llama.generate: prefix-match hit


###Answer
The management of sepsis in a critical care unit involves aggressive fluid resuscitation, antibiotics, surgical excision or drainage of infected tissues and pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Monitoring includes measurement of vital signs, quantification of all fluid intake and output, daily weight, BP by automated sphygmomanometer, and pulse oximetry. Continuous monitoring may be provided by complex devices requiring special training. Symptoms such as fever, tachycardia, tachypnea, confusion, hypotension, oliguria, and organ failure require specific interventions. Diagnosis is clinical, with evaluation for infection through history, physical examination, and tests including urinalysis, urine culture, serial blood cultures, and cultures of other suspect body fluids. Other causes of shock should also be sought. 



### Observations - Fine-tuning the RAG pipeline
We compared **5 combinations** that vary **chunking**, **retriever `k`**, and **LLM parameters**:
- **chunk_size 1000 vs 1500:** larger chunks pack more context into each piece (fewer, longer chunks) but can mix several topics together; smaller chunks are more focused but may split a topic across pieces.
- **Small k (3):** focused and fast - usually the best balance for this manual. **Larger k (5):** more context, but also more noise that can dilute the answer and risk overflowing the context window.
- **temperature 0:** the most factual and repeatable setting (best for medical content). Higher temperature (0.3-0.5) reads more naturally but is less reliable for facts.
- **Recommended setting:** `chunk_size = 1000`, `chunk_overlap = 100`, `k = 3`, `temperature = 0`, `max_tokens ~= 512`.
- *(Tip from the FAQ: if building the vector store is slow, lower `chunk_overlap` to reduce the number of chunks, or raise `chunk_size`.)*

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [34]:
# GROUNDEDNESS judge: does the answer stick to the retrieved context,
# without adding facts that are not supported by it?
groundedness_rater_system_message = """You are an evaluator that rates how GROUNDED an AI answer is.
An answer is "grounded" when every claim it makes is supported by the provided context,
and it does not add information that is missing from the context.
Rate groundedness on a scale of 1 to 5:
  1 = not grounded at all (the answer is mostly invented or contradicts the context)
  5 = fully grounded (every claim is clearly supported by the context)
First output the rating as "Rating: X/5", then give a one-sentence justification.
"""

In [35]:
# RELEVANCE judge: does the answer actually address the user's question?
relevance_rater_system_message = """You are an evaluator that rates how RELEVANT an AI answer is.
An answer is "relevant" when it directly addresses what the user asked,
without drifting to unrelated topics.
Rate relevance on a scale of 1 to 5:
  1 = not relevant (the answer ignores or misunderstands the question)
  5 = fully relevant (the answer directly and completely addresses the question)
First output the rating as "Rating: X/5", then give a one-sentence justification.
"""

In [36]:
# The template that feeds the context, question and answer to the judge.
# It must contain {context}, {question} and {answer} because the
# evaluation function fills these in with .format().
user_message_template = """###Context
{context}

###Question
{question}

###Answer to be evaluated
{answer}
"""

In [37]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [38]:
g, r = generate_ground_relevance_response(query_1, k=3, max_tokens=256)
print("--- GROUNDEDNESS ---")
print(g)
print("\n--- RELEVANCE ---")
print(r)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


--- GROUNDEDNESS ---
 Rating: 5/5
The answer is fully grounded as every claim made in the answer is clearly supported by the context provided. The steps outlined in the answer are directly taken from the text and do not add any missing information.

--- RELEVANCE ---
 Rating: 5/5
The answer directly addresses the question by outlining the steps for managing sepsis in a critical care unit, which aligns with the information provided in the context.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [39]:
g, r = generate_ground_relevance_response(query_2, k=3, max_tokens=256)
print("--- GROUNDEDNESS ---")
print(g)
print("\n--- RELEVANCE ---")
print(r)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


--- GROUNDEDNESS ---
 Rating: 5/5
The answer accurately summarizes the common symptoms of appendicitis from the context and correctly states that it cannot be cured via medicine alone, requiring surgical removal. The answer also specifies the type of surgery (open or laparoscopic appendectomy) and the need for IV antibiotics before the procedure.

--- RELEVANCE ---
 Rating: 5/5
The answer directly addresses the user's question by listing the common symptoms of appendicitis and clarifying that it cannot be cured with medicine alone, instead requiring surgical removal. The answer also specifies the type of surgery (appendectomy) and the need for antibiotics before the procedure.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [40]:
g, r = generate_ground_relevance_response(query_3, k=3, max_tokens=256)
print("--- GROUNDEDNESS ---")
print(g)
print("\n--- RELEVANCE ---")
print(r)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


--- GROUNDEDNESS ---
 Rating: 5/5
The answer accurately summarizes the information provided in the context regarding the causes and treatments for Alopecia Areata, making no additional unsupported claims.

--- RELEVANCE ---
 Rating: 5/5
The answer directly addresses the user's question by providing a list of effective treatments for addressing sudden patchy hair loss, which is a common symptom of Alopecia Areata, and also explains the possible cause behind it as an autoimmune disorder.


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [41]:
g, r = generate_ground_relevance_response(query_4, k=3, max_tokens=256)
print("--- GROUNDEDNESS ---")
print(g)
print("\n--- RELEVANCE ---")
print(r)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


--- GROUNDEDNESS ---
 Rating: 5/5
The answer is fully grounded as every claim made is clearly supported by the context provided. The context outlines various treatments for physical injuries to the brain tissue, which are explicitly stated and detailed in the answer. Additionally, the answer acknowledges that the recommended treatments for brain injuries may be similar to those for spinal injuries mentioned in the text, which is also a valid observation based on the context.

--- RELEVANCE ---
 Rating: 5/5
The answer directly and completely addresses the question by listing out the various treatments recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, based on the context provided.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [42]:
g, r = generate_ground_relevance_response(query_5, k=3, max_tokens=256)
print("--- GROUNDEDNESS ---")
print(g)
print("\n--- RELEVANCE ---")
print(r)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


--- GROUNDEDNESS ---
 Rating: 5/5
The answer is fully grounded as every claim made is clearly supported by the context provided in the text. The text discusses the necessary precautions and treatment steps for a person with a fractured leg, including assessing the severity of the injury, controlling bleeding, immobilizing the leg, managing pain, seeking medical attention, follow-up care, and rehabilitation. The answer directly references these steps and does not add any unsupported information.

--- RELEVANCE ---
 Rating: 5/5
The answer directly addresses the user's question about the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip. It covers assessing the injury, controlling bleeding, immobilizing the leg, managing pain, seeking medical attention, follow-up care, and rehabilitation.


### Observations - Output Evaluation (LLM-as-a-judge)
We used the **same model as a judge** to score each RAG answer on two axes:
- **Groundedness** - is every claim supported by the retrieved context? High scores confirm the RAG guardrails are working and the model is not inventing facts.
- **Relevance** - does the answer actually address the question that was asked?
- Using an LLM as a judge is fast and scalable, but it is **not perfect** - the model can be over-generous when grading itself. For a production system these automatic scores should be **spot-checked by a medical expert**.
- Overall, grounded + relevant scores being high is strong evidence that the RAG system is fit for assisting (not replacing) clinicians.

## Actionable Insights and Business Recommendations

The goal was to build a **RAG-based medical assistant** that answers clinical questions using the trusted *Merck Manual of Diagnosis & Therapy*, instead of relying on an LLM's unverified memory.

**Key takeaways for the business**

1. **A raw LLM is not safe for clinical use.** On its own the model answers from memory, cannot cite a source, and can state wrong facts with full confidence - an unacceptable risk in healthcare.

2. **Prompt engineering helps, but does not solve grounding.** A clear system message and `temperature = 0` make answers more structured and cautious, yet the model still draws on memory and can still hallucinate.

3. **RAG is the key upgrade.** By retrieving relevant passages from the manual and forcing the model to answer only from them, answers become **grounded, traceable, and consistent**, and the system honestly says when the manual does not cover a topic.

4. **Retrieval quality drives answer quality.** Tuning showed that `k = 3` chunks with `temperature = 0` and `~512` output tokens gave the best balance of completeness and factual reliability for this manual.

5. **Built-in evaluation builds trust.** The groundedness and relevance scores give an automatic, repeatable quality check - a foundation for monitoring the assistant over time.

**Business recommendations**

- **Deploy RAG, not a bare LLM.** Use the grounded RAG configuration as the basis of the assistant so every answer is backed by the manual.
- **Position it as a clinician *assistant*, not a replacement.** It speeds up access to trusted knowledge; final decisions stay with the professional. Show the source pages alongside each answer so staff can verify quickly.
- **Keep the knowledge base current.** Re-embed the manual whenever a new edition is released, and consider adding hospital-specific protocols and approved drug formularies as extra sources.
- **Add a human-in-the-loop review** for high-risk areas (e.g. critical care, dosing) and **monitor** the groundedness/relevance scores to catch quality drops early.
- **Expected benefits:** faster bedside decisions, fewer errors from information overload, more standardised care, and reduced time spent searching long reference texts - especially valuable in time-critical situations.

<font size=6 color='blue'>Power Ahead</font>
___